# 12. Object Detection - YoloV5S

| 항목 | 내용 |
|---|---|
| 대상 모델 | `YoloV5S.dxnn` |
| 카테고리 | **Object Detection** |
| 기준 환경 | **DX-All-Suite 2.3.0 / Sync 흐름** |

## 학습 목표

1. 후보 한 줄의 85개 값이 어떤 항목으로 나뉘는지 자리별로 설명할 수 있다.
2. 최종 점수를 두 값의 곱으로 계산하고 두 임계값이 각각 무엇을 걸러내는지 구분할 수 있다.
3. 설정 파일의 key 이름이 코드와 다를 때 값이 적용되지 않음을 근거와 함께 지적할 수 있다.
4. 부류를 구분하지 않는 겹침 정리가 어떤 상황에서 문제가 되는지 설명할 수 있다.

## 1. Object Detection (객체 검출)

**영상에서 정해진 부류에 속하는 대상을 모두 찾아 위치·크기·부류를 함께 답하는 작업**

얼굴 검출이 한 부류만 다루는 데 비해 이 작업은 **무엇인지까지 함께 판정**한다. 그래서 결과 한 건에 사각형과
부류 번호와 점수가 함께 붙는다.

**출력의 성격**

개수가 정해져 있지 않은 목록이다. 같은 대상을 여러 후보가 가리키므로 겹친 것을 정리해야 하고, 대상이 서로
겹쳐 있으면 정리 과정에서 진짜 대상이 지워질 수 있다.

**활용 사례**

- 교통 영상의 차량·보행자 검출
- 재고·불량 개수 세기
- 뒤에 붙는 인식 모델에 넣을 영역 잘라내기

**포함되지 않는 작업**

- 대상 윤곽을 픽셀 단위로 나누기 (Instance Segmentation)
- 같은 대상을 다른 화면에서 다시 찾기 (Person Re-ID)
- 사진 한 장에 이름표 하나 (Image Classification)

## 2. 전체 파이프라인

```text
sample_street.jpg  768×576 BGR
  → 전처리         BGR→RGB · 비율 유지 축소 640×480 · 위아래 여백 80 → 640×640
  → run()          입력 Tensor 하나를 전달
  → npu_0          [1, 255, 80·40·20 격자] 3개 (내부 전달, Public 아님)
  → cpu_0          output0  [1, 25200, 85]  float32  (Public Output — decode 까지 끝난 후보 목록)
  → 후처리         obj 임계값 → 부류 최고점 → conf = obj × cls → 중심좌표를 모서리로 → NMS
  → Result         DetectionResult (box · confidence · class_id)
  → 시각화         사각형과 `이름: 점수` 라벨을 원본 위에 그림
```

![12. Object Detection 전체 파이프라인](resources/12_Object_Detection_pipeline.png)

## 3. 입력과 출력

### 3.1 입력

| 속성 | 값 |
|---|---|
| Tensor 이름 | `images` |
| Shape / Dtype | `[1, 640, 640, 3]` / `uint8` |
| 채널 순서 | NHWC |
| Color / 값 범위 | RGB / 0~255 |
| Resize 방식 | **Letterbox — 비율 유지 후 여백 채우기** |
| Normalization | 없음 |

좌표를 답하는 작업이므로 눌러 넣지 않고 비율을 유지한다. 되돌리는 데 필요한 값은 아래 셋이다.

| 값 | 이번 실행 | 뜻 |
|---|---:|---|
| `gain` | 0.833333 | 축소 비율. `min(640/576, 640/768)` |
| 줄인 크기 | 640×480 | `768 × 0.8333 = 640`, `576 × 0.8333 = 480` |
| `pad_y` | 80 | 위쪽 여백. 아래쪽도 80 |
| `pad_x` | 0 | 가로는 꽉 차서 여백이 없음 |

### 3.2 출력

이 모델은 Task 가 둘이며 **격자 좌표로 되돌리는 계산이 CPU Task 안에서 끝난다.**

| 생성 Task | Tensor | shape / dtype | 다음 사용자 | Public |
|---|---|---|---|---|
| `npu_0` | `Reshape_329` · `_367` · `_405` | `[1, 255, 80·40·20 격자]` / `float32` | `cpu_0` | 아님 — 내부 전달 |
| `cpu_0` | `output0` | `[1, 25200, 85]` / `float32` | Host 후처리 | CPU Task 출력 (= Public 출력) |

`Public` 은 `InferenceEngine.run()` 이 돌려주는 값인지를 가리키는 표시다. 격자 형태의 중간 값은 밖으로 나오지
않고, 후보가 한 줄씩 늘어선 `output0` 만 손에 들어온다.

**후보가 25,200개인 이유**

| stride | 격자 | 자리당 후보 | 후보 수 |
|---:|---|---:|---:|
| 8 | 80×80 | 3 | 19,200 |
| 16 | 40×40 | 3 | 4,800 |
| 32 | 20×20 | 3 | 1,200 |

세 벌을 더하면 `19200 + 4800 + 1200 = 25,200` 이다. 중간 tensor 의 채널 255 도 같은 구조에서 나온다 —
`3 × 85 = 255` 로, 자리당 후보 3개가 각자 85개 값을 갖는다.

**한 줄 85개 값의 자리**

| index | 개수 | 내용 |
|---:|---:|---|
| 0~3 | 4 | 중심 좌표와 크기 `cx, cy, w, h` (입력 640 기준 화소) |
| 4 | 1 | 그 자리에 대상이 있을 점수 (objectness) |
| 5~84 | 80 | 부류마다의 점수 |

**값이 이미 사람이 읽을 수 있는 상태다.** 이번 실행에서 좌표는 `-1.15`~`638.19` 로 입력 640 안에 있고, `obj`
와 부류 점수는 `0`~`0.99` 로 0~1 구간이다. 즉 격자 좌표 복원과 sigmoid 가 CPU Task 안에서 이미 끝났고,
**Host 후처리는 걸러내고 정리하는 일만 한다.**

**최종 점수는 두 값의 곱이다.**

```text
conf = obj × max(class scores)
```

대상이 있을 확신과 그것이 어느 부류인지의 확신을 함께 반영한다. 이번 실행의 1위는 `obj = 0.9227` 과
`cls = 0.9934` 로 `conf = 0.9166` 이다. 부류 번호는 80개 점수 중 가장 큰 자리다.

**80개 부류의 번호**

번호의 정본은 실행 코드가 들고 있는 이름 배열이다. COCO 공식 번호는 1~90 으로 띄엄띄엄한데, 여기서 쓰는
0~79 는 **그 목록에서 실제로 쓰는 80개만 순서대로 눌러 담은 관행**이다. 두 체계를 섞으면 이름이 밀린다.

| index | 이름 | 고른 이유 |
|---:|---|---|
| 0 | person | 목록의 첫 항목 |
| 1 | bicycle | 이번 실행 검출 |
| 2 | car | 이번 실행 검출 |
| 3 | motorcycle | 1·2 와 이어지는 교통 부류 |
| 9 | traffic light | 도로 장면에 흔한 부류 |
| 16 | dog | 이번 실행 1위 |
| 39 | bottle | 중간 자리 확인용 |
| 56 | chair | 중간 자리 확인용 |
| 77 | teddy bear | 끝에서 세 번째 |
| 78 | hair drier | 끝에서 두 번째 |
| 79 | toothbrush | 목록의 마지막 항목 |

전체 목록: [ultralytics coco.yaml](https://raw.githubusercontent.com/ultralytics/ultralytics/main/ultralytics/cfg/datasets/coco.yaml)

#### `.dxnn` 파일에서 직접 확인

`parse_model` 은 `.dxnn` 안의 tensor 이름·shape·dtype·크기·Task 구성을 그대로 보고한다. `npu_0` 이 채널 255
격자 셋을 내고 `cpu_0` 이 `[1, 25200, 85]` 하나로 합치는지 아래 출력에서 확인한다.

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "assets" / "models").is_dir() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
MODEL_PATH = REPO_ROOT / "assets" / "models" / "YoloV5S.dxnn"
assert MODEL_PATH.is_file(), f"모델이 없다: {MODEL_PATH}"

os.environ.setdefault("DX_ALL_SUITE", os.path.expanduser("~/Workspace/dx-all-suite"))
DX_ALL_SUITE = Path(os.environ["DX_ALL_SUITE"])

DX_APP = DX_ALL_SUITE / "dx-runtime" / "dx_app"
assert DX_APP.is_dir(), f"dx-all-suite 가 없다: {DX_APP} — 환경변수 DX_ALL_SUITE 로 위치를 알려 준다."

parsed = subprocess.run(["parse_model", "-m", str(MODEL_PATH), "-v"], capture_output=True, text=True)
print((parsed.stdout + parsed.stderr).strip())

### 3.3 Config.json

| key | 현재 값 | 기본값 | 의미 |
|---|---:|---:|---|
| `obj_threshold` | 0.25 | 0.25 | 대상이 있을 점수가 이 값에 못 미치는 후보를 먼저 버림 |
| `score_threshold` | 0.25 | — | **이 후처리가 읽지 않는 key** |
| `nms_threshold` | 0.45 | 0.45 | 겹침이 이 비율을 넘으면 낮은 점수를 버림 |

**`score_threshold` 는 적용되지 않는다.** 이 후처리가 설정에서 찾는 이름은 `conf_threshold` 이고 파일에는 그
key 가 없다. 그래서 최종 점수 임계값은 **코드 기본값 `0.3`** 이 쓰인다. 파일에 적힌 `0.25` 와 값이 비슷해
겉으로는 문제가 드러나지 않지만, 파일을 고쳐도 결과가 바뀌지 않는다는 뜻이다.

설정으로 동작을 바꾸려 할 때 **key 이름이 코드와 같은지 먼저 확인해야 하는 이유**가 여기 있다. 이 노트북은
원본과 같은 동작을 보이기 위해 `0.3` 을 쓴다.

## 4. 실습

### 4.1 실행 환경 구성

라이브러리·그림 출력 방식·입력 파일 경로를 정하고 설정 파일과 COCO 80개 이름을 읽는다. 이후 셀은 여기서 만든
이름을 그대로 쓴다.

In [ ]:
import json
import sys
import time
from dataclasses import dataclass, field

import matplotlib

# 노트북에서는 결과 그림이 셀 아래 남도록 inline, 스크립트로 단독 실행할 때는 화면이 없으므로
# 파일 기반 Agg 를 쓴다. 지정하지 않으면 GUI 백엔드를 찾다가 실패한다.
try:
    get_ipython().run_line_magic("matplotlib", "inline")  # noqa: F821
except NameError:
    matplotlib.use("Agg")

# 렌더링 백엔드를 dx_engine 보다 먼저 적재한다. matplotlib 은 첫 그림을 그릴 때에야 백엔드를
# 불러오는데, 그 시점이 dx_engine 적재 뒤이면 ft2font 확장 초기화가 실패한다
# (RuntimeError: module 'matplotlib.ft2font' has no attribute '__path__').
import matplotlib.backends.backend_agg  # noqa: E402,F401
import matplotlib.pyplot as plt  # noqa: E402

import cv2  # noqa: E402
import numpy as np  # noqa: E402
from dx_engine import InferenceEngine  # noqa: E402

IMAGE_PATH = DX_APP / "sample" / "img/sample_street.jpg"
CONFIG_PATH = DX_APP / "src" / "python_example" / "object_detection" / "yolov5s" / "config.json"
CONFIG = json.loads(CONFIG_PATH.read_text(encoding="utf-8")) if CONFIG_PATH.is_file() else {}

# dx-all-suite 는 읽기 전용 원본이다. 바이트코드 캐시를 남기지 않도록 먼저 꺼 둔다.
sys.dont_write_bytecode = True
sys.path.insert(0, str(DX_APP / "src" / "python_example"))
from common.utility import get_labels  # noqa: E402

COCO_LABELS = get_labels("coco80")

print("config", CONFIG)
print("classes", len(COCO_LABELS), "/ index 16 =", COCO_LABELS[16])

#### 결과 객체와 실행 지원 함수

- `PreprocessContext` — 전처리가 원본을 어떻게 바꾸었는지 후처리에 넘기는 값. `scale`(gain)·`pad_x`·`pad_y` 가
  좌표 복원의 근거다
- `DetectionResult` — 후처리가 만들어 시각화로 넘기는 결과. 사각형·점수·부류 번호를 담는다
- `notebook_describe_array` — 배열의 shape·dtype·값 범위 출력
- `notebook_show_bgr` — 이미지 inline 표시. 3채널은 RGB 로 되돌림(안 하면 색이 뒤집힘), grayscale 은
  값 범위를 0~255 로 고정(안 하면 대비가 실제보다 강해짐)

In [ ]:
@dataclass
class PreprocessContext:
    """전처리가 원본을 어떻게 바꾸었는지 후처리에 넘기는 값. 원본 대응: common/base/i_processor.py"""

    original_width: int = 0
    original_height: int = 0
    input_width: int = 0
    input_height: int = 0
    scale: float = 1.0
    pad_x: int = 0
    pad_y: int = 0


@dataclass
class DetectionResult:
    """대상 하나의 사각형·점수·부류. 원본 대응: common/base/i_processor.py"""

    box: list = field(default_factory=list)
    confidence: float = 0.0
    class_id: int = 0
    class_name: str = ""


def notebook_describe_array(name: str, array: np.ndarray) -> None:
    """배열의 shape·dtype·값 범위를 한 줄로 출력한다. 노트북 전용."""
    array = np.asarray(array)
    print(f"{name:16s} {str(array.shape):18s} {array.dtype!s:8s} "
          f"min={array.min():9.4f} max={array.max():9.4f} mean={array.mean():9.4f}")


def notebook_show_bgr(image: np.ndarray, title: str = "Image", figsize: tuple = (9, 6)) -> None:
    """BGR 또는 grayscale 이미지를 inline 으로 표시한다. 노트북 전용."""
    plt.figure(figsize=figsize)
    if image.ndim == 2:
        plt.imshow(image, cmap="gray", vmin=0, vmax=255)
    else:
        plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis("off")
    plt.show()

#### 런타임이 보고하는 입출력 정보

`.dxnn` 을 NPU 에 올리고 런타임이 보고하는 입출력 정보를 읽는다. 출력이 `output0` 하나뿐인 것을 확인한다.

In [ ]:
engine = InferenceEngine(str(MODEL_PATH))
input_info = engine.get_input_tensors_info()
output_info = engine.get_output_tensors_info()

print("input :", input_info)
print("output:", output_info)

INPUT_HEIGHT, INPUT_WIDTH = input_info[0]["shape"][1], input_info[0]["shape"][2]
print(f"model input (WxH) = {INPUT_WIDTH}x{INPUT_HEIGHT}")

### 4.2 전처리

비율을 유지해 640×480 으로 줄이고 위아래에 80씩 회색 여백을 넣어 640×640 을 만든다. `gain` 과 여백을 Context 에
적어 두어야 후처리가 좌표를 원본으로 되돌릴 수 있다.

확인할 것:

- 색 순서 — BGR → RGB
- 크기 — 640×640, 그중 실제 사진은 640×480
- 여백 — 위아래 80씩, 좌우 0
- Context — `scale`(gain) 과 `pad_x`·`pad_y`

#### 입력 이미지 확인

원본 사진을 읽어 크기와 dtype 을 확인한다.

In [ ]:
image = cv2.imread(str(IMAGE_PATH))

print(f"shape {image.shape} dtype {image.dtype}")
notebook_show_bgr(image, f"Input - {IMAGE_PATH.name}", figsize=(10, 7))

#### 전처리 함수 정의

- `yolov5_preprocessor_create` — 모델 입력 크기와 여백 색을 state 에 담음
- `yolov5_letterbox` — 비율 유지 축소와 여백 채우기
- `yolov5_preprocessor_process` — letterbox 를 적용하고 복원용 값을 Context 에 담음

In [ ]:
def yolov5_preprocessor_create(input_width: int, input_height: int,
                               pad_color: tuple = (114, 114, 114)) -> dict:
    """모델 입력 크기와 고정 상수를 state 에 담는다. 원본 대응: Factory.create_preprocessor()"""
    return {"input_width": input_width, "input_height": input_height, "pad_color": pad_color}


def yolov5_letterbox(state: dict, image: np.ndarray) -> tuple:
    """비율을 유지해 줄이고 남는 자리를 채운다. 원본 대응: LetterboxPreprocessor._letterbox()"""
    height, width = image.shape[:2]
    gain = min(state["input_height"] / height, state["input_width"] / width)
    new_size = int(round(width * gain)), int(round(height * gain))

    delta_w = (state["input_width"] - new_size[0]) / 2
    delta_h = (state["input_height"] - new_size[1]) / 2
    if (width, height) != new_size:
        image = cv2.resize(image, new_size, interpolation=cv2.INTER_LINEAR)

    top, bottom = int(round(delta_h - 0.1)), int(round(delta_h + 0.1))
    left, right = int(round(delta_w - 0.1)), int(round(delta_w + 0.1))
    padded = cv2.copyMakeBorder(image, top, bottom, left, right, cv2.BORDER_CONSTANT,
                                value=state["pad_color"])
    return padded, gain, (top, left)


def yolov5_preprocessor_process(state: dict, input_image: np.ndarray) -> tuple:
    """이미지를 입력 Tensor 와 Context 로 바꾼다. 원본 대응: Runner.preprocess()"""
    ctx = PreprocessContext()
    ctx.original_height, ctx.original_width = input_image.shape[:2]
    ctx.input_width = state["input_width"]
    ctx.input_height = state["input_height"]

    rgb = cv2.cvtColor(input_image, cv2.COLOR_BGR2RGB)
    padded, gain, pad = yolov5_letterbox(state, rgb)
    ctx.scale = gain
    ctx.pad_y, ctx.pad_x = pad
    return padded, ctx

#### 전처리 실행

입력 Tensor 와 Context 를 만들고 여백이 들어간 모양을 확인한다.

In [ ]:
preprocessor_state = yolov5_preprocessor_create(INPUT_WIDTH, INPUT_HEIGHT)
input_tensor, ctx = yolov5_preprocessor_process(preprocessor_state, image)

print(f"input_tensor  shape={input_tensor.shape} dtype={input_tensor.dtype} "
      f"contiguous={input_tensor.flags['C_CONTIGUOUS']}")
print(f"context       original={ctx.original_width}x{ctx.original_height} "
      f"gain={ctx.scale:.6f} pad_x={ctx.pad_x} pad_y={ctx.pad_y}")

notebook_show_bgr(cv2.cvtColor(input_tensor, cv2.COLOR_RGB2BGR),
                  f"Letterboxed {INPUT_WIDTH}x{INPUT_HEIGHT} (gray bands are padding)",
                  figsize=(6, 6))

### 4.3 추론

#### DXNN Inference 실행

입력 Tensor 를 넘기고 Public Output 을 받는다. 출력되는 시간은 1회 측정값이고 런타임과의 왕복 비용이 포함돼
있어 초당 처리 장수로 환산하지 않는다.

In [ ]:
started = time.perf_counter()
outputs = engine.run([input_tensor])
inference_ms = (time.perf_counter() - started) * 1000.0

print(f"inference: {inference_ms:.2f} ms")
print("output count =", len(outputs))
for index, output in enumerate(outputs):
    array = np.asarray(output)
    print(index, array.shape, array.dtype, f"min={array.min():.4f}", f"max={array.max():.4f}")

#### Raw Output 직접 검사

값에서 직접 확인할 것:

- 후보 수와 항목 수 — `[25200, 85]`
- 좌표 범위 — 입력 640 안이면 격자 복원이 이미 끝난 것
- `obj` 와 부류 점수 범위 — 0~1 이면 sigmoid 가 이미 적용된 것
- 임계값 통과 개수 — 25,200개 중 몇 개인지

In [ ]:
raw = np.squeeze(np.asarray(outputs[0]))
notebook_describe_array("output0", raw)

print(f"후보 {raw.shape[0]:,}개 × 항목 {raw.shape[1]}개 = 5 + {len(COCO_LABELS)}")
print(f"좌표 cx,cy,w,h  {raw[:, :4].min():8.2f} ~ {raw[:, :4].max():8.2f} "
      f"(입력 {INPUT_WIDTH} 안이면 격자 복원 완료)")
print(f"obj            {raw[:, 4].min():8.6f} ~ {raw[:, 4].max():8.6f}")
print(f"부류 점수       {raw[:, 5:].min():8.6f} ~ {raw[:, 5:].max():8.6f}")

obj_threshold = CONFIG.get("obj_threshold", 0.25)
print(f"\nobj >= {obj_threshold} 인 후보 {int((raw[:, 4] >= obj_threshold).sum())}개 / {raw.shape[0]:,}개")
print("첫 후보 앞 8개 값:", np.array2string(raw[0][:8], precision=4))

### 4.4 후처리

후보 목록을 두 번 걸러 내고 겹친 것을 정리한 뒤 좌표를 원본으로 되돌린다. 순서는 `obj` 걸러내기 → 부류
최고점 고르기 → `conf = obj × cls` → 중심좌표를 모서리 좌표로 → NMS → 여백 제거·나누기이며 이 순서를 바꾸지
않는다.

**두 임계값이 걸러내는 것이 다르다.** `obj_threshold` 는 "여기에 뭔가 있는가"만 보고 25,200개를 크게 줄인다.
그다음 `conf_threshold` 는 "그것이 이 부류인가"까지 반영한 최종 점수로 다시 걸러낸다. 앞의 것을 먼저 하는
이유는 계산량 때문이다 — 부류 80개의 최댓값을 25,200번 구하지 않아도 된다.

**겹침 정리가 부류를 구분하지 않는다.** 이 코드의 NMS 는 부류 번호를 인자로 받지만 실제로는 모든 사각형을 한꺼번에
비교한다. 그래서 **사람이 자전거를 타고 있는 장면처럼 서로 다른 부류가 크게 겹치면 한쪽이 지워질 수 있다.**
부류별로 따로 정리하는 방식도 있으며, 어느 쪽을 쓰는지가 결과 개수를 바꾼다.

#### 후처리 함수 정의

- `yolov5_postprocessor_create` — 두 임계값과 부류 수를 state 에 담음
- `yolov5_cxcywh_to_corners` — 중심 좌표와 크기를 모서리 좌표로 변환
- `yolov5_postprocessor_process` — 걸러내기·NMS·좌표 복원을 거쳐 Result 생성

In [ ]:
def yolov5_postprocessor_create(config: dict) -> dict:
    """Config 와 고정 상수를 state 에 담는다. 원본 대응: Factory.create_postprocessor()

    conf_threshold 는 설정 파일에 없는 key 이므로 코드 기본값 0.3 이 그대로 쓰인다.
    """
    return {
        "obj_threshold": config.get("obj_threshold", 0.25),
        "conf_threshold": config.get("conf_threshold", 0.3),
        "nms_threshold": config.get("nms_threshold", 0.45),
        "num_classes": config.get("num_classes", 80),
    }


def yolov5_cxcywh_to_corners(boxes: np.ndarray) -> np.ndarray:
    """중심 좌표와 크기를 모서리 좌표로 바꾼다. 원본 대응: nms_utils.cxcywh_to_x1y1x2y2()"""
    corners = np.zeros_like(boxes)
    corners[:, 0] = boxes[:, 0] - boxes[:, 2] * 0.5
    corners[:, 1] = boxes[:, 1] - boxes[:, 3] * 0.5
    corners[:, 2] = boxes[:, 0] + boxes[:, 2] * 0.5
    corners[:, 3] = boxes[:, 1] + boxes[:, 3] * 0.5
    return corners


def yolov5_postprocessor_process(state: dict, outputs: list, ctx: PreprocessContext) -> list:
    """Raw Output 을 Result 로 변환한다. 원본 대응: Runner.postprocess()"""
    output = np.squeeze(np.asarray(outputs[0]))

    objectness = output[:, 4]
    kept = objectness >= state["obj_threshold"]
    if not np.any(kept):
        return []

    filtered = output[kept]
    class_scores = filtered[:, 5:5 + state["num_classes"]]
    class_best = class_scores.max(axis=1)
    class_ids = class_scores.argmax(axis=1)
    confidences = objectness[kept] * class_best

    boxes = yolov5_cxcywh_to_corners(filtered[:, :4])
    boxes_xywh = np.column_stack([boxes[:, 0], boxes[:, 1],
                                  boxes[:, 2] - boxes[:, 0], boxes[:, 3] - boxes[:, 1]])
    indices = cv2.dnn.NMSBoxes(boxes_xywh.tolist(), confidences.tolist(),
                               state["conf_threshold"], state["nms_threshold"])
    if len(indices) == 0:
        return []

    gain = max(ctx.scale, 1e-6)
    results = []
    for index in np.array(indices).reshape(-1):
        box = boxes[index]
        results.append(DetectionResult(
            box=[float(np.clip((box[0] - ctx.pad_x) / gain, 0, ctx.original_width - 1)),
                 float(np.clip((box[1] - ctx.pad_y) / gain, 0, ctx.original_height - 1)),
                 float(np.clip((box[2] - ctx.pad_x) / gain, 0, ctx.original_width - 1)),
                 float(np.clip((box[3] - ctx.pad_y) / gain, 0, ctx.original_height - 1))],
            confidence=float(confidences[index]),
            class_id=int(class_ids[index]),
        ))
    return results

#### 후처리 실행

확인할 것:

- 단계별 개수 — 25,200 → `obj` 통과 → 최종
- 각 결과의 부류·점수·사각형 — 원본 크기 안에 들어오는지
- 점수의 구성 — `obj × cls` 가 최종 점수와 같은지
- 임계값을 바꿨을 때 — 결과 개수가 어떻게 달라지는지

In [ ]:
postprocessor_state = yolov5_postprocessor_create(CONFIG)
results = yolov5_postprocessor_process(postprocessor_state, outputs, ctx)

obj_pass = int((raw[:, 4] >= postprocessor_state["obj_threshold"]).sum())
print(f"후보 {raw.shape[0]:,}개 → obj {postprocessor_state['obj_threshold']} 통과 {obj_pass}개 "
      f"→ conf {postprocessor_state['conf_threshold']} · NMS {postprocessor_state['nms_threshold']} "
      f"통과 {len(results)}개")

for order, det in enumerate(sorted(results, key=lambda r: -r.confidence), start=1):
    x1, y1, x2, y2 = det.box
    print(f"  #{order} {COCO_LABELS[det.class_id]:12s} conf={det.confidence:.4f} "
          f"box=({x1:6.1f}, {y1:6.1f}, {x2:6.1f}, {y2:6.1f})")

for threshold in (0.1, 0.3, 0.5, 0.9):
    state = yolov5_postprocessor_create({**CONFIG, "conf_threshold": threshold})
    print(f"conf_threshold={threshold} → {len(yolov5_postprocessor_process(state, outputs, ctx))}개")

### 4.5 시각화

#### Visualizer 코드

결과 객체만 받아 사각형과 `이름: 점수` 라벨을 원본 위에 그린다. **원본은 부류마다 색을 무작위로 뽑으므로 실행할
때마다 색이 달라진다.** 노트북에서는 같은 그림이 남도록 난수 씨앗을 고정한다. 그림에 넣는 글자는 영문만 쓴다 —
OpenCV 기본 폰트에 한글 자모가 없어 한글을 넣으면 글자가 깨진다.

In [ ]:
def yolov5_visualizer_create(labels: list, seed: int = 0) -> dict:
    """부류별 색을 만들어 state 에 담는다. 원본 대응: Factory.create_visualizer()"""
    generator = np.random.default_rng(seed)
    return {"labels": labels, "palette": generator.uniform(0, 255, size=(len(labels), 3))}


def yolov5_visualizer_visualize(state: dict, image: np.ndarray, results: list) -> np.ndarray:
    """Result 를 원본 이미지 위에 그린다. 원본 대응: Runner.visualize()"""
    output = image.copy()

    for det in results:
        x1, y1, x2, y2 = [int(value) for value in det.box]
        color = state["palette"][det.class_id % len(state["palette"])]
        cv2.rectangle(output, (x1, y1), (x2, y2), color, 2)

        name = (state["labels"][det.class_id] if det.class_id < len(state["labels"])
                else f"class_{det.class_id}")
        label = f"{name}: {det.confidence:.2f}"
        (label_w, label_h), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
        label_y = y1 - 10 if y1 - 10 > label_h else y1 + 10
        cv2.rectangle(output, (x1, label_y - label_h), (x1 + label_w, label_y + label_h),
                      color, cv2.FILLED)
        cv2.putText(output, label, (x1, label_y), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1,
                    cv2.LINE_AA)

    return output

#### 결과 시각화

검출 결과를 inline 으로 표시한다. 사각형이 대상에 맞고 이름이 맞는지 확인한다.

In [ ]:
visualizer_state = yolov5_visualizer_create(COCO_LABELS)
output_image = yolov5_visualizer_visualize(visualizer_state, image, results)
notebook_show_bgr(output_image, f"{len(results)} objects detected", figsize=(11, 8))

#### 겹침 정리 전의 후보를 함께 보기

최종 결과만 보면 겹침 정리가 무엇을 했는지 보이지 않는다. `obj` 를 통과한 후보 전부를 얇은 선으로 함께 그리면
같은 대상에 후보가 여러 개 몰려 있었다는 사실이 드러난다.

In [ ]:
objectness = raw[:, 4]
kept = objectness >= postprocessor_state["obj_threshold"]
candidate_boxes = yolov5_cxcywh_to_corners(raw[kept][:, :4])

overlay = image.copy()
gain = ctx.scale
for box in candidate_boxes:
    x1 = int(np.clip((box[0] - ctx.pad_x) / gain, 0, ctx.original_width - 1))
    y1 = int(np.clip((box[1] - ctx.pad_y) / gain, 0, ctx.original_height - 1))
    x2 = int(np.clip((box[2] - ctx.pad_x) / gain, 0, ctx.original_width - 1))
    y2 = int(np.clip((box[3] - ctx.pad_y) / gain, 0, ctx.original_height - 1))
    cv2.rectangle(overlay, (x1, y1), (x2, y2), (0, 200, 255), 1)

print(f"obj 통과 후보 {len(candidate_boxes)}개를 노란 선으로, 최종 {len(results)}개를 두꺼운 선으로 그린다.")
notebook_show_bgr(yolov5_visualizer_visualize(visualizer_state, overlay, results),
                  "Candidates before NMS (thin) vs final (thick)", figsize=(11, 8))